# Vivian · 中英混读 tiny-voca 训练
在「运行时 → 更改运行时类型」选择 GPU，再按顺序执行。数据来自 MyDrive/tiny-voca-zh/downloads/vivian-bilingual.zip。默认使用完整 26,910 条清单，从零训练单音色模型。首次准备需要提取约 28 小时音频的基频和时长，可能跨多个 Colab 会话。建议本地运行盘和 Drive 各预留 25 GB 以上。重连后按顺序重跑，已完成的 128 条分块与训练检查点可复用。

双语前端保留中文声调和英文重音。MMS_FA 在整句音频中同时对齐拼音音节与英文单词，再估算这些单元内部的音素时长。切换处、缩写和单词内部的边界需要结合试听判断。[对齐模型说明](https://docs.pytorch.org/audio/2.8.0/generated/torchaudio.pipelines.Wav2Vec2FABundle.html)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, json, collections
drive.mount('/content/drive')
HOME_DIR = Path('/content/drive/MyDrive/tiny-voca-zh')
ARCHIVE = HOME_DIR/'downloads/vivian-bilingual.zip'
WORK = Path('/content/tiny-voca-vivian')
WORK.mkdir(exist_ok=True)
DATA = WORK/'vivian-bilingual'
OUTPUT = HOME_DIR/'vivian-colab'
OUTPUT.mkdir(exist_ok=True)
SMOKE = False  # 首次检查可设为 True，从三个语种各取少量样本。
STEPS = 80 if SMOKE else 104000
BATCH_FRAMES = 4000
MAX_BATCH = 16
assert ARCHIVE.is_file(), ARCHIVE
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], check=True)
print(ARCHIVE, round(ARCHIVE.stat().st_size/1e9,2),'GB')

## 安装独立训练环境
使用 Python 3.11、配套的 Torch 和 TorchAudio 2.8，以及 eSpeak NG。依赖安装在运行时本地。

In [ ]:
def run(*args, **kwargs):
    subprocess.run(list(map(str,args)),check=True,**kwargs)
run('apt-get','update','-qq')
run('apt-get','install','-qy','espeak-ng')
run(sys.executable,'-m','pip','install','-q','uv')
PY = WORK/'env/bin/python'
if not PY.exists():
    run('uv','venv','--python','3.11',WORK/'env')
run('uv','pip','install','--python',PY,'--index-url','https://download.pytorch.org/whl/cu128','torch==2.8.0','torchaudio==2.8.0')
run('uv','pip','install','--python',PY,'numpy==2.2.6','librosa==0.11.0','soundfile==0.13.1','pypinyin==0.55.0','num2words==0.5.14')
run(PY,'-c','import torch,torchaudio; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name())')

## 训练代码与双语前端
笔记本内置 tiny-voca-zh 版本 7ec7434 所需模块，并在延迟读取特征时关闭无限增长的内存缓存。模型源码与双语前端会随检查点保存在 Drive。

In [ ]:
import base64, io
SRC = WORK/'source'
SRC.mkdir(exist_ok=True)
bundle = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle))) as z:
    z.extractall(SRC)
os.environ['PYTHONPATH']=str(SRC)
os.environ['TINY_VOCA_FRONTEND']='espeak'

In [ ]:
(SRC/'tiny_voca/text.py').write_text(r'''
"""Vivian Mandarin/English frontend shared by annotation and synthesis."""
import hashlib
import re
import unicodedata
from functools import lru_cache
from pathlib import Path
import pypinyin
from . import zh_text as zh, en_text as en, context_en

LANGUAGE = "mixed"
DEFAULT_SPEAKERS = ("Vivian",)
FRONTENDS = ("vivian-mixed",)
FRONTEND_VARIABLE = "VIVIAN_FRONTEND"
NothingToPronounce = zh.NothingToPronounce
SYMBOLS = zh.SPECIALS + zh.PUNCTUATION + tuple("zh/"+p for p in zh.PHONES) + tuple("en/"+p for p in en.PHONES)
SYMBOL_TO_ID = {p:i for i,p in enumerate(SYMBOLS)}
PHONE_IDS = {i for i,p in enumerate(SYMBOLS) if "/" in p}
VOWELS = tuple("zh/"+p for p in zh.VOWELS) + tuple("en/"+p for p in en.VOWELS)
def set_frontend(name):
    if name not in FRONTENDS:
        raise ValueError(name)
def frontend_identity():
    return dict(phonemizer=FRONTENDS[0], symbols=list(SYMBOLS))
@lru_cache()
def frontend_fingerprint():
    h = hashlib.sha256()
    for module in (__file__, zh.__file__, en.__file__, context_en.__file__):
        h.update(Path(module).read_bytes())
    h.update((pypinyin.__version__ + en.espeak_version()).encode())
    return h.hexdigest()

@lru_cache(maxsize=16000)
def english(word):
    phones = en.phonemize(word)
    if phones.unknown:
        raise ValueError(f"Unmapped English phoneme in {word}")
    return [(("en/"+p if p in en.PHONE_SET else p), s) for p,s in zip(phones.symbols[1:-1],phones.stress[1:-1])]

def units(text):
    from num2words import num2words
    text = unicodedata.normalize("NFKC", text).replace("’","'")
    han = r"[\u3400-\u9fff]"
    if re.search(han,text):
        text = zh.normalize_text(re.sub(r"[-−](?=\d)", "负", text))
    else:
        text = re.sub(r"-?\d+(?:\.\d+)?",lambda m:num2words(m.group()),text)
    english_overrides = context_en.context_phones(text)
    tokens = re.finditer(han+r"+|[A-Za-z]+(?:'[A-Za-z]+)*|[^\s]",text)
    out = [("",[(zh.BOS,0)])]
    for token in tokens:
        part = token.group()
        if re.fullmatch(han+"+",part):
            readings = zh.phrase_readings(part)
            syllables = zh.phrase_syllables(part, readings)
            if len(readings) != len(part) or None in syllables:
                raise ValueError(f"Unmapped Mandarin syllable in {part}")
            for i, (reading, sy) in enumerate(zip(readings, syllables)):
                if (sy.char == '儿' and i and part[i-1:i+1] in ('会儿', '块儿', '那儿', '玩儿')
                        and not part.startswith(('儿童', '儿科', '儿子', '儿女', '儿歌', '儿孙'), i)):
                    # Rhotacization is a coda of the previous alignment unit.
                    # Reuse the existing retroflex token, without an extra vowel.
                    spelling, phones = out[-1]
                    out[-1] = (spelling + 'r', phones + [('zh/r', 0)])
                    continue
                phones = ([sy.initial] if sy.initial else []) + [sy.final+str(sy.tone)]
                spelling = reading.rstrip('12345').replace('ü', 'u').replace('v', 'u')
                out.append((spelling, [('zh/'+p, 0) for p in phones]))
        elif re.match("[A-Za-z]",part):
            if out[-1][0]:
                out.append(("",[(zh.SP,0)]))
            out.append((part.lower().replace("'",""), english_overrides.get(token.start()) or english(part)))
        elif part in zh.PUNCTUATION:
            out.append(("",[(part,0)]))
    if not any(word for word,_ in out):
        raise NothingToPronounce(text)
    return out+[("",[(zh.EOS,0)])]

def phonemize(text):
    pairs=[pair for _,phones in units(text) for pair in phones]
    symbols,stress=zip(*pairs)
    return zh.Phonemized(symbols,tuple(SYMBOL_TO_ID[s] for s in symbols),stress,())
'''.lstrip(), encoding='utf-8')


In [ ]:
(SRC/'preprocess.py').write_text(r'''
"""Align each recording using MMS words/syllables and cache small chunks on Drive."""
import collections, hashlib, json, os, shutil, sys, tarfile
from pathlib import Path
import torch, torchaudio
from torchaudio.pipelines import MMS_FA
from tiny_voca.config import Config
from tiny_voca.prepare import _prepare_row
from tiny_voca.text import units, phonemize, frontend_fingerprint

data, local, drive = map(Path,sys.argv[1:4])
c=Config(speakers=("Vivian",))
local.mkdir(parents=True,exist_ok=True)
key=hashlib.sha256()
for name in ("train","validation","test"):
    key.update((data/(name+".jsonl")).read_bytes())
key.update((frontend_fingerprint()+c.fingerprint).encode())
key.update(Path(__file__).read_bytes())
tag=key.hexdigest()
cache=drive/"preparation"/tag
cache.mkdir(parents=True,exist_ok=True)
model=None

@torch.inference_mode()
def durations(text,wave,frames):
    global model
    if model is None:
        model=MMS_FA.get_model().eval().cuda()
    groups=units(text)
    words=[w for w,_ in groups if w]
    x=torchaudio.functional.resample(wave,24000,16000).unsqueeze(0).cuda()
    emission,_=model(x)
    labels=MMS_FA.get_tokenizer()(words)
    flat=[v for group in labels for v in group]
    if not all(labels) or len(flat)+sum(a==b for a,b in zip(flat,flat[1:]))>emission.size(1):
        raise ValueError("CTC target exceeds emission length")
    spans=iter(MMS_FA.get_aligner()(emission[0],labels))
    scale=frames/emission.size(1)
    centers=[]
    voiced=[]
    for word,phones in groups:
        if word:
            span=next(spans)
            lo,hi=span[0].start*scale,span[-1].end*scale
            for j,(symbol,_) in enumerate(phones):
                centers.append(lo+(j+.5)*(hi-lo)/len(phones))
                voiced.append("/" in symbol)
        else:
            centers.append(None)
            voiced.append(False)
    # Anchor punctuation/boundaries in available gaps. Each pronounced phone gets >= 1 frame.
    if sum(voiced)>frames: raise ValueError("Too many phones")
    centers[0],centers[-1]=0.,float(frames)
    import numpy as np
    indices=[i for i,v in enumerate(centers) if v is not None]
    curve=np.interp(np.arange(len(centers)),indices,[centers[i] for i in indices])
    cuts=np.rint((curve[:-1]+curve[1:])/2).astype(int)
    result=[]
    pos=0
    remaining=sum(voiced)
    for i,cut in enumerate([*cuts,frames]):
        remaining-=voiced[i]
        end=min(frames-remaining,max(pos+int(voiced[i]),int(cut)))
        result.append(end-pos)
        pos=end
    return result

def dump(file,rows):
    file.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in rows))

torch.set_num_threads(1)
for split in ("train","validation","test"):
    source=data/(split+".jsonl")
    rows=list(map(json.loads,source.read_text().splitlines()))
    (local/split).mkdir(exist_ok=True)
    accepted=[]; rejected=[]
    for offset in range(0,len(rows),128):
        label=f"{split}-{offset:05d}"
        archive=cache/(label+".tar")
        good=local/(label+".jsonl"); bad=local/(label+".rejected.jsonl")
        if archive.exists():
            with tarfile.open(archive) as t: t.extractall(local,filter="data")
        else:
            batch=[]; failures=[]
            for i,row in enumerate(rows[offset:offset+128],offset):
                try:
                    ph=phonemize(row["text"])
                    if len(ph)>c.max_tokens: raise ValueError("Too many tokens")
                    record=_prepare_row(source,0,i,row,c,"text","voice_id",split,local/split)
                    wave=torch.load(local/record["path"],weights_only=True)["wave"]
                    d=durations(row["text"],wave,record["frames"])
                    assert len(d)==len(ph) and sum(d)==record["frames"]
                    record.update(tokens=ph.ids,stress=ph.stress,durations=d,language=row["language"],
                                  frontend=frontend_fingerprint(),annotation=tag)
                    batch.append(record)
                except ValueError as exc:
                    failures.append(dict(id=row["id"],language=row["language"],reason=str(exc)))
            dump(good,batch); dump(bad,failures)
            tmp=archive.with_suffix(".partial")
            with tarfile.open(tmp,"w") as t:
                for path in [good,bad,*[local/r["path"] for r in batch]]:
                    t.add(path,arcname=path.relative_to(local))
            os.replace(tmp,archive)
        accepted.extend(map(json.loads,good.read_text().splitlines()))
        rejected.extend(map(json.loads,bad.read_text().splitlines()))
        print(split,offset+min(128,len(rows)-offset),"/",len(rows),"accepted",len(accepted),flush=True)
    dump(local/(split+".jsonl"),accepted)
    dump(cache/(split+".rejected.jsonl"),rejected)
    for lang in ("zh","en","mixed"):
        n=sum(r["language"]==lang for r in rows)
        used=sum(r["language"]==lang for r in accepted)
        print(split,lang,used,"/",n)
        if not n or used<.8*n: raise ValueError("Inspect rejection log before training")
(local/"annotations.json").write_text(json.dumps(dict(complete=True,fingerprint=tag,acoustic_root=str(local))))
'''.lstrip(), encoding='utf-8')

## 验证数据与混读分词

In [ ]:
import hashlib
with zipfile.ZipFile(ARCHIVE) as z:
    # 从 ZIP 中央目录生成版本标记，Drive 上换包后重新解压。
    fingerprint=hashlib.sha256(str([(i.filename,i.CRC,i.file_size) for i in z.infolist()]).encode()).hexdigest()
    marker=WORK/'archive-id'
    if not marker.exists() or marker.read_text()!=fingerprint:
        z.extractall(WORK)
        marker.write_text(fingerprint)
seen=set()
for split in ('train','validation','test'):
    rows=list(map(json.loads,(DATA/(split+'.jsonl')).read_text().splitlines()))
    assert all(r['split']==split and r['voice_id']=='Vivian' and (DATA/r['audio']).is_file() for r in rows)
    keys={r['text_id'] for r in rows}
    assert not seen.intersection(keys)
    seen.update(keys)
    print(split,collections.Counter(r['language'] for r in rows))
run(PY,'-c', '''from tiny_voca.text import phonemize
p=phonemize("最低气温是-10度，请打开 Google Colab。")
assert any(s.startswith("en/") for s in p.symbols) and any(s.startswith("zh/") for s in p.symbols)
print(*p.symbols)''')

## 提取特征与强制对齐
每块完成即存到 Drive。重连时会恢复到本地磁盘。按语种输出接受量；被拒样本的 ID 和原因保存在 preparation 下。

In [ ]:
if SMOKE:
    subset=WORK/'vivian-smoke'
    subset.mkdir(exist_ok=True)
    if not (subset/'audio').exists():
        (subset/'audio').symlink_to(DATA/'audio', target_is_directory=True)
    shutil.copy2(DATA/'prepare-config.json',subset/'prepare-config.json')
    for split in ('train','validation','test'):
        rows=list(map(json.loads,(DATA/(split+'.jsonl')).read_text().splitlines()))
        count=8 if split=='train' else 2
        picked=[r for lang in ('zh','en','mixed') for r in [v for v in rows if v['language']==lang][:count]]
        (subset/(split+'.jsonl')).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in picked))
    INPUT_DATA=subset
else:
    INPUT_DATA=DATA
FEATURES=WORK/('features-smoke' if SMOKE else 'features')
run(PY,SRC/'preprocess.py',INPUT_DATA,FEATURES,OUTPUT)

## 训练与续训
默认每 500 步保存到 Drive。改变 batch 或其他训练配方时应使用新实验名。中途重连后重跑前面的单元，再运行本单元即可续训。

In [ ]:
import shutil
annotation=json.loads((FEATURES/'annotations.json').read_text())['fingerprint']
experiment=OUTPUT/('model-'+annotation[:12])
experiment.mkdir(exist_ok=True)
shutil.copytree(SRC,experiment/'source',dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
RUN=experiment/'checkpoints'
last=RUN/'last.pt'
args=[PY,'-m','tiny_voca.train','--config',DATA/'prepare-config.json','--device','cuda',
      '--train',FEATURES/'train.jsonl','--validation',FEATURES/'validation.jsonl',
      '--output',RUN,'--steps',STEPS,'--batch-frames',BATCH_FRAMES,'--max-batch',MAX_BATCH,
      '--save-every',500,'--keep-every',10000,'--max-seconds',86400,'--no-preload']
if last.exists():
    args.extend(['--resume',last])
def saved_step():
    if not last.exists(): return 0
    return int(subprocess.check_output([str(PY),'-c','import torch,sys; print(torch.load(sys.argv[1],map_location="cpu",weights_only=True)["step"])',str(last)]))
if saved_step()<STEPS:
    if not last.exists() and RUN.exists() and all(f.name in ('recipe.json','config.json','metrics.jsonl') for f in RUN.iterdir()):
        from datetime import datetime
        RUN.rename(experiment/('interrupted-before-first-save-'+datetime.now().strftime('%Y%m%d%H%M%S')))
    run(*args)
print('已保存步数',saved_step(),RUN)

## 用检查点试听
可在训练中断并已有 inference.pt 时单独运行。下列三句使用同一个模型和同一个 Vivian 音色。

In [ ]:
from IPython.display import Audio, display
annotation=json.loads((FEATURES/'annotations.json').read_text())['fingerprint']
checkpoint=OUTPUT/('model-'+annotation[:12])/'checkpoints/inference.pt'
for lang,text in [('zh','今天最低气温是负十度。'),('en','Could you open the window, please?'),('mixed','这个 weekend 我想去 hiking，你来吗？')]:
    wav=WORK/(lang+'.wav')
    run(PY,'-m','tiny_voca.infer','--checkpoint',checkpoint,'--voice','Vivian','--text',text,'--output',wav)
    print(text)
    display(Audio(filename=str(wav)))